[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 06](README.md)

# CUDA: coalescencia, memoria compartida y tiling

**Tema:** 06 · **Sesiones:** 27, 28 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo aumentar reutilización sin exceder recursos ni romper bordes y sincronización?


## Resultados de aprendizaje

- Relacionar coalescencia, bancos y memoria compartida.
- Calcular recursos de un tile.
- Validar GEMM tiled frente a CPU para dimensiones irregulares.


## Modelo conceptual

Tiling carga datos reutilizados en memoria compartida y sincroniza antes de consumirlos.

Un tile mayor puede aumentar reutilización pero también registros, memoria compartida y presión de ocupación.

Las dimensiones no múltiplos requieren cargas condicionadas y ceros fuera del dominio.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "06"
NOTEBOOK = "06_cuda/02_memoria_tiling.ipynb"
assert (ROOT / "curso" / "notebooks" / "06_cuda" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Recursos por tile

Se calculan hilos, bloques, memoria compartida y tiles de K para GEMM.


In [ ]:
def tile_resources(m, n, k, tile, bytes_per_value=4):
    return {
        "grid": ((n+tile-1)//tile, (m+tile-1)//tile),
        "threads_per_block": tile*tile,
        "shared_bytes": 2*tile*tile*bytes_per_value,
        "k_tiles": (k+tile-1)//tile,
    }
for tile in (8, 16, 32):
    row = tile_resources(1000, 777, 513, tile)
    assert row["threads_per_block"] <= 1024
    print(tile, row)


**Interpretación.** La validez geométrica no garantiza buena ocupación; se consulta el límite real y el perfil del kernel.


## Intensidad aproximada

Se compara reutilización ideal de una GEMM ingenua y una tiled.


In [ ]:
def intensity(tile, bytes_per_value=4):
    flops = 2 * tile * tile * tile
    bytes_loaded = 2 * tile * tile * bytes_per_value
    return flops / bytes_loaded
for tile in (8, 16, 32): print(tile, f"{intensity(tile):.2f} FLOP/byte por fase ideal")
assert intensity(32) > intensity(8)


**Interpretación.** El cálculo ideal omite escrituras, cachés, bordes y recargas; sirve para formular la hipótesis de reutilización.


## Práctica reproducible

1. Comparar GEMM ingenua, tiled y cuBLAS con la misma precisión.
2. Incluir dimensiones no divisibles por tile.
3. Perfilar coalescencia, bancos, ocupación y tiempo total.


## Errores frecuentes

- Sincronizar fuera de una rama que no alcanza todo el bloque.
- Comparar operaciones diferentes.
- Elegir tile solo por tiempo de un caso.

## Criterios de aceptación

- Bordes comprobados con referencia CPU.
- Recursos por bloque dentro de límites.
- Interpretación sustentada en métricas del perfil.


## Referencias y material relacionado

- [Guía CUDA](README.md)
- [Fuentes CUDA heredadas](../../../cuda/03_cuda_thread_programming/)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 06](README.md)
